<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/02-data-preprocessing-feature-engineering.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Data Representation, Preprocessing, and Feature Engineering**

A learning algorithm never observes the real world directly. It receives a finite table, tensor, collection of documents, or stream of events constructed from measurements. The decisions made while constructing that representation determine which relationships are easy to learn, which distinctions disappear, and which accidental shortcuts become available. For this reason, data preparation is not merely a cleanup step before the "real" modeling begins. It is part of the model's **inductive bias**.

The full path is:

> real-world process -> observations -> samples and labels -> cleaned representation -> learned transformations -> model-ready features

Each arrow introduces assumptions. A row may be treated as independent even though several rows belong to one patient. A missing value may be replaced by a median even though the act of missingness carries information. A category may be assigned an integer even though the numbers create a false order. A scaler may be fitted on the complete dataset and silently reveal test-set statistics. Good preprocessing makes these assumptions explicit and preserves the boundary between information available during training and information reserved for evaluation.

This chapter develops a practical vocabulary for reasoning about those choices. It begins with dataset construction and quality, then connects distance measures to feature scaling, covers numerical and categorical transformations, separates feature engineering from selection and extraction, treats imbalance as a decision problem, and ends with split-aware preprocessing pipelines.


### **From Real-World Observations to a Dataset**

Before choosing an encoder or imputer, define the **unit of analysis**: what does one sample represent? In a house-price model it may be one sale; in medical prediction it may be one patient at a particular decision time; in fraud detection it may be one transaction; and in image classification it may be one image. This definition controls what counts as an independent example, which metadata must stay together during splitting, and when the target becomes observable.

#### **Samples, Features, Targets, and Metadata**

For a supervised dataset with $n$ samples and $d$ model inputs, the conventional notation is

$$
X \in \mathbb{R}^{n \times d}, \qquad
y = (y_1,\ldots,y_n).
$$

The $i$-th row $x_i$ is one sample and the $j$-th column is one feature. The target $y_i$ is the outcome that the model should predict. This notation is convenient, but real datasets also contain identifiers, timestamps, source systems, annotator IDs, group IDs, and audit fields. Such **metadata** may be essential for splitting, debugging, or monitoring while being inappropriate as a model input.

| Role | Question it answers | Example | Common mistake |
|---|---|---|---|
| Sample | What is one prediction about? | one customer at renewal time | treating repeated visits as independent people |
| Feature | What is known at prediction time? | tenure before renewal | including information recorded after the outcome |
| Target | What event or value should be predicted? | churn within 30 days | using an ambiguous or inconsistently labelled proxy |
| Metadata | What helps govern the dataset? | customer ID, hospital, timestamp | either feeding IDs to the model or discarding them before split design |
| Group/time key | Which rows must remain related? | patient ID or event time | randomly mixing related or future observations across folds |

A useful discipline is to write a **prediction contract** before building $X$: at time $t$, for entity $e$, predict outcome $Y$ over horizon $h$, using only information available by $t$. This single sentence catches many forms of target leakage.

#### **Structured, Unstructured, and Multimodal Data**

**Structured data** has an explicit schema: columns have defined meanings and types. **Unstructured data** such as text, images, audio, and raw logs needs a representation step before most estimators can use it. **Semi-structured data**, including JSON records and event logs, has fields but often lacks a fixed rectangular schema. **Multimodal data** combines several sources, such as an image, a clinical note, and laboratory measurements for the same case.

The distinction matters because different representations preserve different information. Turning a document into word counts discards word order; resizing an image changes spatial detail; aggregating a month of transactions into a mean and maximum discards event sequence. The resulting feature vector is not a neutral copy of reality. It is a compressed answer to the question, "Which aspects of this observation might matter for this task?"

#### **Sampling, Coverage, and Representativeness**

The training set is a sample from a **data-generating process**, not simply a large collection of rows. A model generalizes when future deployment examples are sufficiently similar to the distribution represented during development. Sample size alone cannot repair systematic coverage gaps: ten million daytime traffic images do not teach a model about night-time glare, and a survey collected only through a smartphone app excludes people who cannot or do not use that app.

Check coverage across target classes, geography, time, acquisition devices, demographic groups, rare operating conditions, and entities that generate many repeated rows. Also distinguish:

- **selection bias**, where inclusion in the dataset depends on factors related to the outcome;
- **survivorship bias**, where failed or missing cases disappear from observation;
- **temporal drift**, where relationships change after the collection period;
- **aggregation bias**, where a pattern learned for the population does not hold for important subgroups.

These are not problems that an optimizer can solve. They require better collection, a narrower claim, stratified analysis, reweighting under defensible assumptions, or explicit uncertainty about unsupported regions.


<details>
<summary><strong>Python example: audit roles, keys, missingness, and validity</strong></summary>

```python
import numpy as np
import pandas as pd

# A tiny raw extract. customer_id and event_time are governance fields,
# while age, city, and plan are candidate model inputs.
raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C03", "C03", "C04", "C05"],
        "event_time": pd.to_datetime(
            ["2026-01-03", "2026-01-06", "2026-01-08",
             "2026-01-08", "2026-01-12", "2026-01-15"]
        ),
        "age": [34, np.nan, 121, 45, 29, 52],
        "city": ["Sydney", "Melbourne", "Sydney", "Sydney", None, "Perth"],
        "plan": ["basic", "pro", "basic", "basic", "pro", "basic"],
        "churn_30d": [0, 1, 0, 0, 1, 0],
    }
)

# Keep separate checks for identity, completeness, and domain validity.
audit = pd.DataFrame(
    {
        "dtype": raw.dtypes.astype(str),
        "missing": raw.isna().sum(),
        "unique_values": raw.nunique(dropna=True),
    }
)

duplicate_entity_time = raw.duplicated(
    subset=["customer_id", "event_time"], keep=False
)
invalid_age = ~raw["age"].between(0, 110) & raw["age"].notna()

print(audit)
print("\nDuplicate entity-time rows:", raw.index[duplicate_entity_time].tolist())
print("Invalid age rows:", raw.index[invalid_age].tolist())
```

</details>


### **Data Quality and Cleaning**

Data cleaning should protect meaning, not merely remove inconvenient rows. The central question is whether a value is absent, impossible, unusual, duplicated, noisy, or genuinely rare. These conditions require different responses. A purchase of one million dollars may be a decimal error, a corporate customer, or the most important event in a fraud dataset.

#### **Missing Values**

A missing entry is both an absent measurement and potentially an observation about the collection process. Three classical mechanisms help organize the reasoning:

| Mechanism | Meaning | Example | Consequence |
|---|---|---|---|
| MCAR | missingness is unrelated to observed or unobserved values | a sensor packet is lost at random | complete-case analysis may be unbiased but wastes data |
| MAR | missingness depends on observed variables | income is omitted more often for a known age group | model missingness using observed context; conditional imputation may be reasonable |
| MNAR | missingness depends on the missing value itself or an unobserved cause | very high earners avoid reporting income | ordinary imputation cannot identify the truth without extra assumptions or data |

Common strategies include deleting a feature with little usable coverage, imputing a numerical median, imputing a categorical mode or explicit `Unknown` category, using model-based imputation, and adding a missingness indicator. The indicator can preserve the fact that a measurement was unavailable, while the imputed value allows an estimator to accept a complete matrix. However, if missingness is caused by a deployment workflow that later changes, a model may over-rely on that operational pattern.

Every imputation statistic is learned state. The median, category vocabulary, and iterative imputation model must be fitted on the training portion only and then applied unchanged to validation and test data.

#### **Outliers and Invalid Records**

An **invalid record** violates a known constraint: negative age, a date before the system existed, or a category outside the data contract. An **outlier** is unusual relative to a distribution but may still be correct. Validation rules can reject or quarantine invalid values; outliers require domain investigation.

Univariate detectors include the interquartile-range rule and robust z-scores based on the median absolute deviation. Multivariate outliers may look ordinary in each feature separately but unusual in combination. Deleting observations solely because they are hard to model biases the dataset. Alternatives include correcting upstream parsing, capping only when the measurement process has known limits, applying robust transformations, choosing robust estimators, or evaluating performance separately on the tail.

#### **Duplicates and Conflicting Labels**

Exact duplicates can arise from repeated exports, retries, or joins. Near-duplicates may be the same document with formatting changes or adjacent frames from one video. If related copies appear on both sides of a split, evaluation measures memorization instead of generalization. Deduplication therefore belongs before final splitting, and entity identifiers or similarity clusters may be needed to keep related samples together.

Conflicting labels require provenance: who labelled the sample, under which guideline, with what confidence, and at what time? Majority vote hides disagreement; probabilistic labels, adjudication, or annotator models may be more appropriate when ambiguity is real.

#### **Measurement Error and Label Noise**

Feature noise changes $x$; label noise changes $y$. Random measurement noise often increases variance and weakens learnable relationships. Systematic measurement error is more dangerous because it can create stable but false patterns. Label noise can be symmetric, class-dependent, or instance-dependent. For example, subtle positive cases may be mislabelled more often than obvious ones.

Useful responses include repeated measurements, sensor calibration, annotation audits, inter-annotator agreement, high-confidence review sets, robust losses, and sensitivity analysis. A model that fits noisy labels perfectly has not necessarily learned the underlying task.


<details>
<summary><strong>Python example: impute values while preserving missingness signals</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

df = pd.DataFrame(
    {
        "age": [22, 31, np.nan, 47, np.nan],
        "income": [52000, np.nan, 68000, 91000, 61000],
        "city": ["Sydney", "Sydney", None, "Perth", "Melbourne"],
    }
)

# Numeric imputation also appends one binary column for every feature that
# contained missing values during fit.
numeric_imputer = SimpleImputer(strategy="median", add_indicator=True)
numeric_ready = numeric_imputer.fit_transform(df[["age", "income"]])

# A categorical mode is a simple baseline. In production, an explicit
# "Unknown" category can be preferable when missingness has distinct meaning.
categorical_imputer = SimpleImputer(strategy="most_frequent")
categorical_ready = categorical_imputer.fit_transform(df[["city"]])

print("Numeric values plus missing indicators:\n", numeric_ready)
print("\nCategorical values:\n", categorical_ready.ravel())
print("\nTraining medians:", numeric_imputer.statistics_)
```

</details>


### **Similarity and Distance Measures**

Many algorithms turn representation into geometry. K-nearest neighbours predicts from nearby samples; k-means assigns points to nearby centroids; kernel methods convert pairwise similarity into a learning signal; retrieval systems rank candidates by similarity. A distance measure therefore states what it means for two examples to be alike.

#### **Euclidean and Manhattan Distance**

For two vectors $x,z \in \mathbb{R}^{d}$, Euclidean distance is

$$
d_2(x,z)=\sqrt{\sum_{j=1}^{d}(x_j-z_j)^2}.
$$

The index $j$ selects a feature, $x_j-z_j$ is the difference along that feature, squaring prevents positive and negative differences from cancelling, and the square root returns the result to the original unit. Large coordinate differences receive disproportionate influence because they are squared.

Manhattan distance is

$$
d_1(x,z)=\sum_{j=1}^{d}|x_j-z_j|.
$$

It adds absolute coordinate differences and is less dominated by one large deviation. Euclidean distance produces spherical neighbourhoods; Manhattan distance produces diamond-shaped neighbourhoods. Neither is automatically correct. Both assume that coordinates are commensurable, which is why feature scaling changes their meaning.

#### **Cosine Similarity**

Cosine similarity compares direction rather than magnitude:

$$
\operatorname{cos}(x,z)=\frac{x^\top z}{\|x\|_2\|z\|_2}.
$$

The numerator $x^\top z$ is the dot product; each denominator is a Euclidean vector length. For non-zero vectors, the ratio is the cosine of the angle between them. Two documents with similar term proportions can therefore be close even if one is much longer. Cosine distance is often defined as $1-\operatorname{cos}(x,z)$, although its exact metric properties depend on the domain and transformation.

#### **Distances for Binary and Mixed Data**

For binary attributes, **Hamming distance** counts positions that differ. **Jaccard similarity** ignores joint absences and compares active sets:

$$
J(A,B)=\frac{|A\cap B|}{|A\cup B|}, \qquad d_J=1-J.
$$

This is useful when a shared absence is not evidence of similarity, such as two users who both did not purchase thousands of products. For mixed numerical, ordinal, and categorical data, Gower distance computes a type-appropriate normalized difference per feature and averages across observed features. It is interpretable, but feature weights and normalization ranges still encode assumptions.

#### **Choosing a Task-Appropriate Similarity**

Ask four questions: Which invariances should similarity preserve? Are feature units comparable? Are zeros meaningful absences? Does high dimensionality make distances concentrate? A distance that looks mathematically familiar can be semantically wrong. For example, integer-encoding cities makes `city=3` twice as far from `city=1` as `city=2`, although those numbers are arbitrary labels.

| Data meaning | Common starting point | Important caution |
|---|---|---|
| continuous measurements on comparable scales | Euclidean | sensitive to scale and large deviations |
| sparse counts or text vectors | cosine | ignores magnitude, which may itself matter |
| binary presence/absence | Jaccard | ignores co-absence by design |
| mixed tabular features | Gower or learned representation | normalization and feature weighting control the result |
| domain-specific sequences or graphs | task-specific metric or embedding | generic vector distances may discard structure |


<details>
<summary><strong>Python example: calculate several notions of similarity</strong></summary>

```python
import numpy as np

x = np.array([2.0, 1.0, 0.0, 3.0])
z = np.array([1.0, 2.0, 0.0, 1.0])

euclidean = np.sqrt(np.sum((x - z) ** 2))
manhattan = np.sum(np.abs(x - z))
cosine = np.dot(x, z) / (np.linalg.norm(x) * np.linalg.norm(z))

# Jaccard treats non-zero coordinates as a set of active attributes.
x_active = set(np.flatnonzero(x))
z_active = set(np.flatnonzero(z))
jaccard = len(x_active & z_active) / len(x_active | z_active)

print(f"Euclidean distance: {euclidean:.3f}")
print(f"Manhattan distance: {manhattan:.3f}")
print(f"Cosine similarity:  {cosine:.3f}")
print(f"Jaccard similarity: {jaccard:.3f}")
```

</details>


### **Numerical Feature Processing**

Numerical columns can represent counts, money, probabilities, ages, durations, coordinates, ranks, or identifiers. Treating them all as interchangeable real numbers can create a misleading geometry. Processing should reflect units, support, skew, uncertainty, and the model family.

#### **Standardization and Normalization**

Standardization transforms feature $j$ using training-set mean $\mu_j$ and standard deviation $\sigma_j$:

$$
x'_{ij}=\frac{x_{ij}-\mu_j}{\sigma_j}.
$$

The transformed training feature has mean near zero and standard deviation near one. Standardization does not make a distribution Gaussian; it only changes location and scale. It is especially important for distance-based methods, gradient-based optimization, principal component analysis, and regularized linear models, because otherwise a large-unit feature can dominate distance or penalty terms.

Min-max scaling maps a feature to a chosen interval, commonly ([0,1]):

$$
x'_{ij}=\frac{x_{ij}-\min(x_j)}{\max(x_j)-\min(x_j)}.
$$

It preserves order but is sensitive to extreme minima and maxima. Robust scaling instead subtracts the median and divides by an interquartile range, reducing the influence of tails. **Vector normalization** is different: it rescales each sample, not each feature, often to unit length. This is useful when direction matters more than magnitude, as in cosine-based text representations.

![K-nearest-neighbour decision boundaries before and after feature scaling. Without scaling, the high-variance feature dominates the geometry.](assets/feature-scaling-knn.png){fig-alt="Two KNN decision surface plots comparing unscaled and standardized features"}

*Source: [scikit-learn, Importance of Feature Scaling](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_scaling_importance.html).*

Tree-based models split one feature at a time and are usually insensitive to monotonic scaling, but preprocessing can still matter when trees are combined with distance, linear, or neural components.


<details>
<summary><strong>Python example: compare feature scaling with row normalization</strong></summary>

```python
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, Normalizer, RobustScaler, StandardScaler

X = pd.DataFrame(
    {
        "age": [20, 30, 40, 50],
        "annual_spend": [1000, 1100, 1200, 20000],  # one large but valid value
    }
)

transformers = {
    "standard": StandardScaler(),
    "min_max": MinMaxScaler(),
    "robust": RobustScaler(),
    "row_l2_normalized": Normalizer(norm="l2"),
}

for name, transformer in transformers.items():
    transformed = transformer.fit_transform(X)
    print(f"\n{name}\n", pd.DataFrame(transformed, columns=X.columns).round(3))
```

</details>

#### **Skewed Variables and Transformations**

Income, transaction size, duration, and word frequency often have long right tails. A logarithmic transformation,

$$
x'=\log(1+x),
$$

compresses large ratios while remaining defined at zero. It is appropriate when multiplicative differences are more meaningful than additive differences. Box-Cox transformations learn a power parameter but require positive values; Yeo-Johnson extends the idea to zero and negative values. Quantile transforms can map ranks toward a uniform or normal reference distribution, but they may distort distances and extrapolate poorly beyond the training range.

A transformation should solve a modeling problem: stabilize variance, make an approximately linear relationship easier to express, reduce outlier leverage, or align the feature with domain scale. It should not be applied merely because a histogram is asymmetric.

#### **Discretization and Interaction Features**

Discretization replaces a continuous variable with intervals. Equal-width bins preserve the measurement scale; quantile bins aim for similar sample counts; domain bins encode meaningful thresholds. Binning can make nonlinear threshold effects accessible to a linear model, but it discards within-bin order and can create brittle boundaries.

An **interaction feature** represents a joint effect. If the impact of advertising depends on season, a term $x_{\text{advertising}}x_{\text{season}}$ lets a linear model express that dependence. Polynomial basis expansions add powers and cross-products, turning a linear estimator in the expanded features into a nonlinear function of the original input. The price is rapidly increasing dimensionality and a greater need for regularization.


<details>
<summary><strong>Python example: transform skew, create bins, and add interactions</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures

df = pd.DataFrame(
    {
        "monthly_spend": [0, 20, 35, 80, 160, 900],
        "tenure_years": [0.2, 1.0, 2.0, 4.0, 6.0, 10.0],
    }
)

# log1p is stable at zero and compresses the long right tail.
df["log_spend"] = np.log1p(df["monthly_spend"])

# Quantile bins illustrate a threshold representation. Fit bin boundaries on
# training data only when this is used in a real model pipeline.
df["spend_band"] = pd.qcut(
    df["monthly_spend"], q=3, labels=["low", "medium", "high"]
)

# Degree 2 adds x1^2, x1*x2, and x2^2. The interaction can express a joint effect.
poly = PolynomialFeatures(degree=2, include_bias=False)
expanded = poly.fit_transform(df[["log_spend", "tenure_years"]])
expanded = pd.DataFrame(expanded, columns=poly.get_feature_names_out())

print(df)
print("\nExpanded features:\n", expanded.round(3))
```

</details>


### **Categorical Feature Processing**

Categories are symbols, not measurements. An encoding must translate them into numbers without inventing unsupported relationships. The right choice depends on whether the category is ordered, how many distinct values exist, whether unseen values are expected, and which estimator will consume the representation.

#### **Ordinal and One-Hot Encoding**

**Ordinal encoding** maps categories to integers according to a real order, such as `low < medium < high`. It is appropriate only when order exists. Even then, the gaps between integer codes are an assumption: coding levels as 0, 1, and 2 suggests equally spaced effects unless a flexible model can learn otherwise.

**One-hot encoding** creates one binary feature per category. It removes false order and allows a linear model to learn a separate coefficient for each value. For $K$ categories, a full one-hot representation has $K$ dimensions and one active coordinate per sample. Unknown categories must receive an explicit policy, such as an out-of-vocabulary bucket or an all-zero vector under `handle_unknown="ignore"`.

![A categorical value is mapped through a vocabulary to an index and then to a sparse one-hot feature vector.](assets/one-hot-encoding-pipeline.svg){fig-alt="Google diagram showing categorical values mapped to IDs and one-hot vectors"}

*Source: [Google Machine Learning Crash Course, Vocabulary and One-Hot Encoding](https://developers.google.com/machine-learning/crash-course/categorical-data/one-hot-encoding?hl=en).*


<details>
<summary><strong>Python example: encode ordered and unordered categories</strong></summary>

```python
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

train = pd.DataFrame(
    {
        "risk_level": ["low", "medium", "high", "medium"],
        "city": ["Sydney", "Perth", "Sydney", "Melbourne"],
    }
)
test = pd.DataFrame(
    {
        "risk_level": ["high"],
        "city": ["Darwin"],  # unseen during fit
    }
)

ordinal = OrdinalEncoder(
    categories=[["low", "medium", "high"]],
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)
one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

train_risk = ordinal.fit_transform(train[["risk_level"]])
train_city = one_hot.fit_transform(train[["city"]])
test_city = one_hot.transform(test[["city"]])

print("Ordinal risk:", train_risk.ravel())
print("One-hot columns:", one_hot.get_feature_names_out().tolist())
print("Training cities:\n", train_city)
print("Unseen test city:\n", test_city)
```

</details>

#### **Target and Frequency Encoding**

Frequency encoding replaces a category $c$ with its training count or proportion. It is compact and does not use labels, but two semantically different categories with equal frequency become indistinguishable.

Target encoding replaces $c$ with a label statistic such as its mean target. For binary classification, a smoothed estimate is

$$
\operatorname{TE}(c)=\frac{n_c\bar{y}_c+\alpha\bar{y}}{n_c+\alpha},
$$

where $n_c$ is the number of training examples in category $c$, $\bar{y}_c$ is their target mean, $\bar{y}$ is the global training mean, and $\alpha$ controls shrinkage toward the global mean. Rare categories receive stronger shrinkage because their local estimate is uncertain.

Naively computing $\bar{y}_c$ on the same rows later used for fitting leaks each row's target into its own feature. Safe training values require **cross-fitting**: calculate encodings for each fold using only the other folds. Validation and test categories are then encoded from the complete training partition. This issue is especially severe for high-cardinality IDs, where an unsmoothed encoding can almost memorize labels.


<details>
<summary><strong>Python example: cross-fitted smoothed target encoding</strong></summary>

```python
import pandas as pd
from sklearn.model_selection import StratifiedKFold

df = pd.DataFrame(
    {
        "city": ["A", "A", "A", "B", "B", "B", "C", "C", "C", "D", "D", "D"],
        "target": [0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1],
    }
)

alpha = 4.0
oof_encoded = pd.Series(index=df.index, dtype=float)
splitter = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)

for train_idx, valid_idx in splitter.split(df[["city"]], df["target"]):
    fold_train = df.iloc[train_idx]
    fold_valid = df.iloc[valid_idx]
    fold_mean = fold_train["target"].mean()

    stats = fold_train.groupby("city")["target"].agg(["count", "mean"])
    mapping = (
        stats["count"] * stats["mean"] + alpha * fold_mean
    ) / (stats["count"] + alpha)

    # A category unseen in this fold falls back to the fold's global mean.
    oof_encoded.iloc[valid_idx] = (
        fold_valid["city"].map(mapping).fillna(fold_mean)
    )

result = df.assign(city_target_encoded=oof_encoded)
print(result)
```

</details>

#### **Feature Hashing and High Cardinality**

One-hot encoding becomes expensive for categories such as URLs, product IDs, and words. **Feature hashing** maps each category through a deterministic hash function into one of $m$ buckets. Memory is bounded and no vocabulary dictionary is required, which suits streaming or very large sparse data. The trade-off is collision: unrelated categories can share a bucket, and the original category cannot be reconstructed from the representation. Increasing $m$ lowers collision frequency at greater memory cost.

Other options for high cardinality include grouping rare categories, learned embeddings, hierarchical domain encodings, or carefully cross-fitted target encoding. Raw identifiers should rarely be treated as ordinary categories unless the deployment goal genuinely includes memorizing known entities.


<details>
<summary><strong>Python example: hash sparse categorical tokens into fixed dimensions</strong></summary>

```python
from sklearn.feature_extraction import FeatureHasher

# Each sample is represented as a collection of categorical tokens.
records = [
    ["city=Sydney", "device=mobile", "plan=basic"],
    ["city=Perth", "device=desktop", "plan=pro"],
    ["city=Melbourne", "device=mobile", "plan=basic"],
]

hasher = FeatureHasher(
    n_features=8,
    input_type="string",
    alternate_sign=False,
)
X_hashed = hasher.transform(records)

print("Shape:", X_hashed.shape)
print("Dense view for teaching:\n", X_hashed.toarray())
```

</details>


### **Feature Engineering, Selection, and Extraction**

Feature engineering changes the representation so that useful regularities become easier for a model to express. Feature selection removes original variables; feature extraction creates a new lower-dimensional representation. These operations can improve accuracy, data efficiency, interpretability, or computation, but they must be learned inside the evaluation procedure.

#### **Domain Features and Basis Expansions**

Domain features translate raw measurements into quantities closer to the mechanism of interest: body-mass index from height and weight, account age from timestamps, click-through rate from clicks and impressions, cyclical sine/cosine features from hour of day, or ratios that normalize for exposure. A good engineered feature captures a stable relationship available at prediction time.

Basis expansions, including polynomial features, splines, radial basis functions, and Fourier components, provide a model with reusable nonlinear building blocks. They can make a simple estimator powerful and interpretable, but they also amplify extrapolation risk. For example, a high-degree polynomial can behave wildly outside the observed range.

#### **Filter, Wrapper, and Embedded Selection**

| Family | How selection is decided | Strength | Limitation |
|---|---|---|---|
| Filter | score each feature using correlation, ANOVA, mutual information, or another statistic | fast and model-independent | often misses interactions and redundancy |
| Wrapper | repeatedly evaluate subsets with an estimator, such as recursive feature elimination | reflects model behaviour | computationally expensive and easy to overfit |
| Embedded | selection occurs while fitting, such as L1 sparsity or tree-based importance | efficient and model-aware | inherits the estimator's assumptions and importance biases |

Selection is supervised when it uses $y$. It must therefore occur separately inside each training fold. Selecting features once on the full dataset and then cross-validating gives the validation labels an opportunity to influence the representation.


<details>
<summary><strong>Python example: compare filter, wrapper, and embedded selection</strong></summary>

```python
from sklearn.datasets import make_classification
from sklearn.feature_selection import RFE, SelectFromModel, SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

X, y = make_classification(
    n_samples=300,
    n_features=12,
    n_informative=4,
    n_redundant=2,
    shuffle=False,
    random_state=7,
)
names = [f"x{i}" for i in range(X.shape[1])]

# Filter: rank features independently using an ANOVA F statistic.
filter_selector = SelectKBest(score_func=f_classif, k=4).fit(X, y)

# Wrapper: repeatedly fit the estimator and remove weaker features.
wrapper_selector = RFE(
    LogisticRegression(max_iter=2000), n_features_to_select=4
).fit(X, y)

# Embedded: an L1 penalty drives some fitted coefficients exactly to zero.
embedded_selector = SelectFromModel(
    LogisticRegression(penalty="l1", solver="liblinear", C=0.15, max_iter=2000)
).fit(X, y)

def selected(mask):
    return [name for name, keep in zip(names, mask) if keep]

print("Filter:  ", selected(filter_selector.get_support()))
print("Wrapper: ", selected(wrapper_selector.get_support()))
print("Embedded:", selected(embedded_selector.get_support()))
```

</details>

#### **Feature Extraction versus Feature Selection**

Selection preserves a subset of original coordinates, so the result can retain names such as `age` and `blood_pressure`. Extraction builds new coordinates. Principal component analysis, for example, creates orthogonal directions that explain variance; neural embeddings learn task-relevant dense vectors; matrix factorization represents users and items through latent factors.

Extraction can compress correlated information and improve geometry, but a component is usually harder to explain than an original variable. It may also preserve high-variance nuisance structure rather than predictive information. The choice is a trade-off among prediction, interpretability, storage, and robustness, not a universal preprocessing rule.


<details>
<summary><strong>Python example: extract principal components after standardization</strong></summary>

```python
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X, _ = load_wine(return_X_y=True)
X_scaled = StandardScaler().fit_transform(X)

# Keep the smallest number of components explaining at least 90% of variance.
pca = PCA(n_components=0.90).fit(X_scaled)
X_reduced = pca.transform(X_scaled)

print("Original shape:", X.shape)
print("Reduced shape: ", X_reduced.shape)
print("Explained variance retained:", round(pca.explained_variance_ratio_.sum(), 3))
```

</details>


### **Imbalanced Data**

A dataset is imbalanced when target classes occur at substantially different frequencies. Imbalance is not automatically a defect: rare disease, fraud, equipment failure, and extreme weather are genuinely uncommon. The problem is that ordinary empirical risk and accuracy may favour the majority class even when minority errors are more costly.

Start with the decision. How costly is a false negative relative to a false positive? Is the deployment prevalence equal to the training prevalence? Do predicted probabilities need calibration? Resampling changes the training distribution, while class weights change the loss; neither choice alone determines the final operating threshold.

#### **Resampling and Synthetic Examples**

Random undersampling reduces majority examples and computation but may discard useful variation. Random oversampling repeats minority examples and preserves them, but repeated points can encourage overfitting. Synthetic Minority Over-sampling Technique (SMOTE) generates a point between a minority sample $x_i$ and a selected minority neighbour $x_j$:

$$
x_{\text{new}}=x_i+\lambda(x_j-x_i), \qquad \lambda\sim U(0,1).
$$

The scalar $\lambda$ chooses a position on the line segment. This fills local minority regions rather than simply copying rows.

![A synthetic minority sample is generated by interpolating between two nearby minority examples.](assets/smote-sample-generation.png){fig-alt="SMOTE interpolation between two minority samples"}

*Source: [imbalanced-learn, Sample Generator Used in SMOTE-like Samplers](https://imbalanced-learn.org/stable/auto_examples/over-sampling/plot_illustration_generation_sample.html).*

SMOTE assumes interpolation is meaningful. It can create implausible combinations for categorical features, bridge separate minority clusters, or generate ambiguous samples near a class boundary. Most importantly, resampling must occur inside each training fold. Creating synthetic samples before splitting can put related information into validation data.


<details>
<summary><strong>Python example: reproduce the core SMOTE interpolation step</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(12)
x_i = np.array([0.25, 3.52])
x_j = np.array([0.43, 3.65])

# Select a random point on the segment joining two minority neighbours.
lam = rng.uniform(0.0, 1.0)
x_new = x_i + lam * (x_j - x_i)

print("lambda:", round(lam, 3))
print("minority point i:", x_i)
print("minority point j:", x_j)
print("synthetic point:  ", x_new.round(3))
```

</details>

#### **Class Weights and Cost-Sensitive Learning**

Weighted loss gives selected mistakes greater influence. A common balanced class weight is

$$
w_c=\frac{N}{K N_c},
$$

where $N$ is the number of training samples, $K$ is the number of classes, and $N_c$ is the number in class $c$. Rare classes receive larger weights. This heuristic balances aggregate class contributions, but domain costs may justify different weights.

Cost-sensitive learning can also use a decision matrix or tune the prediction threshold on validation data. Weighting frequently raises minority recall at the cost of precision. That trade-off should be measured with precision-recall curves, class-specific metrics, expected cost, and calibration rather than accuracy alone.


<details>
<summary><strong>Python example: compare unweighted and class-weighted classifiers</strong></summary>

```python
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=2000,
    n_features=12,
    n_informative=5,
    weights=[0.95, 0.05],
    class_sep=1.0,
    random_state=4,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=4
)

models = {
    "unweighted": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000)
    ),
    "balanced": make_pipeline(
        StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000)
    ),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(
        name,
        {
            "recall": round(recall_score(y_test, pred), 3),
            "precision": round(precision_score(y_test, pred), 3),
            "balanced_accuracy": round(balanced_accuracy_score(y_test, pred), 3),
        },
    )
```

</details>


### **Data Splitting and Leakage Prevention**

The split is a simulation of future use. A good split withholds exactly the information that will be unavailable when the model is deployed. Random row splitting is appropriate only when rows are exchangeable and independent enough for that simulation.

#### **Random, Grouped, Temporal, and Stratified Splits**

| Split strategy | Preserves or withholds | Appropriate example | Failure avoided |
|---|---|---|---|
| Random | random sample of rows | independent manufactured parts from a stable process | accidental ordering effects |
| Stratified | approximate class proportions | rare-event classification | folds with no or very few positive examples |
| Grouped | complete entities or clusters | patients, users, schools, videos | memorizing an entity seen in another fold |
| Temporal | future observations | demand forecasting or changing fraud patterns | training on information from the future |
| Spatial/source holdout | locations or acquisition domains | new hospital or camera | overstating transfer to unseen environments |

Stratification and grouping solve different problems. A stratified random split can still leak patients across folds. `StratifiedGroupKFold` tries to satisfy both class-balance and group-separation constraints, but exact balance may be impossible.

![TimeSeriesSplit uses expanding past training windows and evaluates on later observations.](assets/time-series-split.png){fig-alt="Scikit-learn TimeSeriesSplit folds with training samples before test samples"}

*Source: [scikit-learn, Visualizing Cross-Validation Behavior](https://scikit-learn.org/stable/auto_examples/model_selection/plot_cv_indices.html).*


<details>
<summary><strong>Python example: inspect random, grouped, and temporal splits</strong></summary>

```python
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, TimeSeriesSplit, train_test_split

n = 24
X = np.arange(n).reshape(-1, 1)
y = np.array([0, 1, 0] * 8)
groups = np.repeat(np.arange(8), 3)  # three rows per entity

# Random and stratified: class proportions are preserved, but groups are ignored.
train_idx, test_idx = train_test_split(
    np.arange(n), test_size=0.25, stratify=y, random_state=2
)
print("Stratified test indices:", np.sort(test_idx))

# Grouped: no entity appears on both sides.
group_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=2)
g_train, g_test = next(group_split.split(X, y, groups))
print("Train groups:", np.unique(groups[g_train]))
print("Test groups: ", np.unique(groups[g_test]))

# Temporal: every test window lies after its corresponding training window.
for fold, (t_train, t_test) in enumerate(TimeSeriesSplit(n_splits=3).split(X), 1):
    print(f"Time fold {fold}: train 0..{t_train[-1]}, test {t_test[0]}..{t_test[-1]}")
```

</details>

#### **Target Leakage and Preprocessing Leakage**

**Target leakage** occurs when a feature contains information about the outcome that would not be available at prediction time. Examples include a discharge code used to predict admission outcome, a refund status used to predict fraud at purchase time, or an aggregate accidentally calculated with future events.

**Preprocessing leakage** occurs when a transformation learns from held-out observations. Fitting a scaler, imputer, feature selector, target encoder, resampler, or vocabulary on all rows makes the representation depend on the evaluation set. The model may never receive `y_test` explicitly, yet its development process has still used unavailable information.

![Incorrect preprocessing learns from all rows before splitting, while the correct workflow splits first and fits every learned transformation on training data.](assets/data-leakage-boundary.svg){fig-alt="Comparison of a leaky preprocessing workflow and a train-only pipeline"}

The safe rule is simple: **split first; fit every data-dependent operation on training data; transform validation and test data without refitting**. During cross-validation, this rule must be repeated inside each fold.


<details>
<summary><strong>Python example: observe feature-selection leakage on random labels</strong></summary>

```python
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline

rng = np.random.default_rng(8)
X = rng.normal(size=(240, 4000))
y = rng.integers(0, 2, size=240)  # deliberately random: no real signal exists

# WRONG: labels from all rows influence which features are selected.
X_leaky = SelectKBest(f_classif, k=20).fit_transform(X, y)
X_train, X_test, y_train, y_test = train_test_split(
    X_leaky, y, test_size=0.30, stratify=y, random_state=8
)
leaky_model = LogisticRegression(max_iter=2000).fit(X_train, y_train)
leaky_accuracy = accuracy_score(y_test, leaky_model.predict(X_test))

# CORRECT: each fold fits the selector using only that fold's training rows.
safe_pipeline = Pipeline(
    [
        ("select", SelectKBest(f_classif, k=20)),
        ("model", LogisticRegression(max_iter=2000)),
    ]
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=8)
safe_scores = cross_val_score(safe_pipeline, X, y, cv=cv, scoring="accuracy")

print("Leaky holdout accuracy:", round(leaky_accuracy, 3))
print("Safe CV accuracy:      ", round(safe_scores.mean(), 3))
print("Chance is approximately 0.5 because the labels are random.")
```

</details>


### **Reproducible Preprocessing Pipelines**

A production-ready preprocessing workflow is a stateful function. During `fit`, it learns medians, means, scales, category vocabularies, selected features, or model parameters. During `transform` and `predict`, it reuses exactly that learned state. The same contract must hold in notebooks, validation, batch inference, and online serving.

#### **Scikit-Learn Pipeline and ColumnTransformer**

`Pipeline` chains operations so that they are fitted in order and invoked consistently. `ColumnTransformer` sends different columns through different branches, such as median imputation plus scaling for numerical features and mode imputation plus one-hot encoding for categorical features. The transformed blocks are concatenated before reaching the estimator.

![A mixed table is divided into numerical and categorical preprocessing branches, recombined, and passed to one estimator.](assets/mixed-feature-pipeline.svg){fig-alt="ColumnTransformer diagram with numerical and categorical branches"}

This structure provides more than convenience:

- cross-validation refits preprocessing within each fold;
- hyperparameter search can tune preprocessing and model choices together;
- inference automatically applies the training transformations;
- the fitted object records one reproducible path from raw schema to prediction;
- column-specific logic is easier to test and inspect.

#### **Fitting Transformations on Training Data Only**

For a train/validation/test design, the workflow is:

1. define the split using entity, time, or sampling constraints;
2. construct an unfitted pipeline;
3. fit it on the training partition;
4. use validation data for model and threshold decisions without refitting preprocessing on validation rows;
5. perform one final evaluation on untouched test data;
6. after evaluation is complete, optionally refit the selected specification on all development data for deployment.

Cross-validation replaces steps 3 and 4 with repeated fold-specific fits. The final test set remains outside that loop. Random seeds, software versions, schema definitions, fitted artifacts, and data snapshots should be recorded so that the same transformation can be reproduced.


<details>
<summary><strong>Python example: build and evaluate a mixed-type preprocessing pipeline</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

rng = np.random.default_rng(21)
n = 500

# Generate a small mixed-type dataset with a known but noisy churn mechanism.
age_complete = rng.integers(18, 76, size=n).astype(float)
income_complete = rng.lognormal(mean=11.0, sigma=0.45, size=n)
tenure = rng.uniform(0, 12, size=n)
city = rng.choice(["Sydney", "Melbourne", "Perth"], size=n, p=[0.5, 0.35, 0.15])
plan = rng.choice(["basic", "pro"], size=n, p=[0.65, 0.35])

logit = (
    -1.6
    + 0.035 * (age_complete - 40)
    - 0.20 * tenure
    + 0.75 * (plan == "basic")
    + 0.35 * (city == "Perth")
)
probability = 1.0 / (1.0 + np.exp(-logit))
y = rng.binomial(1, probability)

# Introduce missing feature values after the target mechanism is generated.
age = age_complete.copy()
income = income_complete.copy()
age[rng.choice(n, 30, replace=False)] = np.nan
income[rng.choice(n, 40, replace=False)] = np.nan

X = pd.DataFrame(
    {
        "age": age,
        "income": income,
        "tenure": tenure,
        "city": city,
        "plan": plan,
    }
)

numeric_features = ["age", "income", "tenure"]
categorical_features = ["city", "plan"]

numeric_pipeline = Pipeline(
    [
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    [
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocess = ColumnTransformer(
    [
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

model = Pipeline(
    [
        ("preprocess", preprocess),
        ("classifier", LogisticRegression(max_iter=2000)),
    ]
)

# The complete pipeline is refitted inside every training fold.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=21)
scores = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring=["roc_auc", "balanced_accuracy"],
)
print("Mean ROC AUC:", round(scores["test_roc_auc"].mean(), 3))
print("Mean balanced accuracy:", round(scores["test_balanced_accuracy"].mean(), 3))

# Fit the selected specification and predict a row containing an unseen city.
model.fit(X, y)
new_customers = pd.DataFrame(
    {
        "age": [31.0, np.nan],
        "income": [72000.0, 61000.0],
        "tenure": [1.2, 7.5],
        "city": ["Darwin", "Sydney"],
        "plan": ["basic", "pro"],
    }
)
print("Predicted churn probabilities:", model.predict_proba(new_customers)[:, 1].round(3))
```

</details>


### **Data Preparation Checklist**

Data preparation is complete when the representation, split, and fitted transformation can be defended together. The following checklist is intentionally ordered: later steps cannot repair a broken prediction contract or unrepresentative sample.

| Stage | Questions to answer | Evidence to retain |
|---|---|---|
| Prediction contract | What entity, decision time, target horizon, and available information define one prediction? | written task definition and feature availability timestamps |
| Sampling | Which population, period, sources, groups, and rare conditions are covered? | collection protocol and subgroup counts |
| Schema | Which fields are features, targets, metadata, IDs, groups, and timestamps? | versioned schema and validation rules |
| Quality | Why are values missing, duplicated, invalid, or extreme? | audit report and documented handling decisions |
| Representation | Which invariances and relationships do scaling, encoding, distance, or engineered features impose? | transformation specification and rationale |
| Imbalance | What error costs and deployment prevalence matter? | class distribution, cost assumptions, threshold policy |
| Split | Does the split simulate future entities, time, and domains without leakage? | saved split keys or deterministic split procedure |
| Pipeline | Is every learned preprocessing step fitted inside training folds and reused at inference? | serialized pipeline and cross-validation code |
| Validation | Are performance, calibration, subgroups, and uncertainty evaluated on untouched data? | evaluation report linked to the dataset version |
| Monitoring | Which schema changes, missingness shifts, category drift, and performance signals will be watched? | production data checks and alert thresholds |

The most important comparison is not "raw data versus clean data." It is **an informal transformation that quietly uses unavailable information versus a documented, train-only pipeline that encodes the intended deployment problem**. A sophisticated model cannot compensate for a representation that deletes the signal, invents false geometry, or contaminates evaluation. Conversely, careful preprocessing can make a simple model accurate, interpretable, and much easier to trust.

The next chapters formalize the geometry, probability, and optimization ideas behind these transformations. Later model chapters will repeatedly return to the same principle: preprocessing, estimator, and evaluation design form one learning system.

[Back to Machine Learning guideline](Machine Learning.html)
